# STEP 2C: Deployment Gates & Safety Checks

## Validation Before Production Deployment

This notebook demonstrates the 5 deployment gates:
1. Input data validation
2. Output score validation
3. Distribution shift checking
4. PSI drift detection
5. Diagnostics validation

## Setup

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
from datetime import datetime

print('[OK] All imports successful')
print(f'Step 2C: Deployment Gates & Safety Checks')

## Gate 1: Input Data Validation

In [ ]:
print('='*70)
print('GATE 1: INPUT DATA VALIDATION')
print('='*70)

# Create test data
test_data = pd.DataFrame({
    'physician_id': [f'PHY{i:05d}' for i in range(100)],
    'specialty': np.random.choice(['Surgery', 'Medicine'], 100),
    'state': np.random.choice(['CA', 'TX', 'NY'], 100),
    'annual_claims': np.random.randint(10, 100, 100),
})

required_cols = ['physician_id', 'specialty', 'state']
missing = [c for c in required_cols if c not in test_data.columns]
null_pct = (test_data[required_cols].isnull().sum().max() / len(test_data)) * 100

checks = {
    'Required columns present': not missing,
    'Null rate < 5%': null_pct < 5.0,
    'No duplicates': test_data['physician_id'].duplicated().sum() == 0,
}

print(f'\nInput Validation Results:')
passed = 0
for check, result in checks.items():
    status = '[PASS]' if result else '[FAIL]'
    print(f'  {check:<35} {status}')
    if result:
        passed += 1

gate1_pass = passed == len(checks)
print(f'\nGate 1 Status: {"[PASS]" if gate1_pass else "[FAIL]"}')

## Gate 2: Output Score Validation

In [ ]:
print('\n' + '='*70)
print('GATE 2: OUTPUT SCORE VALIDATION')
print('='*70)

# Create scores
scores = np.random.uniform(1, 10, 100)

score_checks = {
    'All scores in [1.0, 10.0]': ((scores >= 1.0) & (scores <= 10.0)).all(),
    'No null scores': not np.isnan(scores).any(),
    'Sufficient variance': np.std(scores) > 0.5,
}

print(f'\nOutput Validation Results:')
passed = 0
for check, result in score_checks.items():
    status = '[PASS]' if result else '[FAIL]'
    print(f'  {check:<35} {status}')
    if result:
        passed += 1

gate2_pass = passed == len(score_checks)
print(f'\nGate 2 Status: {"[PASS]" if gate2_pass else "[FAIL]"}')

## Gate 3: Distribution Shift Checking

In [ ]:
print('\n' + '='*70)
print('GATE 3: DISTRIBUTION SHIFT CHECKING')
print('='*70)

baseline_mean = 5.5
baseline_std = 1.5
current_mean = np.mean(scores)
current_std = np.std(scores)

mean_shift_pct = abs((current_mean - baseline_mean) / baseline_mean) * 100
std_shift_pct = abs((current_std - baseline_std) / baseline_std) * 100

shift_threshold = 5.0

print(f'\nDistribution Shift Analysis:')
print(f'  Baseline mean: {baseline_mean:.2f}, Current: {current_mean:.2f}')
print(f'  Shift: {mean_shift_pct:.1f}% (threshold: {shift_threshold}%)')
print(f'  Status: {"[ALERT]" if mean_shift_pct > shift_threshold else "[OK]"}')
print(f'\n  Baseline std: {baseline_std:.2f}, Current: {current_std:.2f}')
print(f'  Shift: {std_shift_pct:.1f}% (threshold: {shift_threshold}%)')
print(f'  Status: {"[ALERT]" if std_shift_pct > shift_threshold else "[OK]"}')

gate3_pass = (mean_shift_pct <= shift_threshold) and (std_shift_pct <= shift_threshold)
print(f'\nGate 3 Status: {"[PASS]" if gate3_pass else "[WARNING]"}' )

## Gate 4: PSI Drift Detection

In [ ]:
print('\n' + '='*70)
print('GATE 4: PSI DRIFT DETECTION')
print('='*70)

def calculate_psi(baseline, current, n_bins=5):
    baseline_counts = np.histogram(baseline, bins=n_bins)[0] + 0.0001
    current_counts = np.histogram(current, bins=n_bins)[0] + 0.0001
    baseline_pct = baseline_counts / baseline_counts.sum()
    current_pct = current_counts / current_counts.sum()
    psi = (current_pct * np.log(current_pct / baseline_pct)).sum()
    return psi

baseline_scores = np.random.normal(5.5, 1.5, 500).clip(1, 10)
psi = calculate_psi(baseline_scores, scores)

psi_threshold = 0.25

print(f'\nPSI Analysis:')
print(f'  PSI Value: {psi:.4f}')
print(f'  Threshold: {psi_threshold:.4f}')
print(f'\n  Interpretation:')
if psi < 0.10:
    print(f'    [OK] No significant change')
elif psi < psi_threshold:
    print(f'    [WARNING] Small change detected')
else:
    print(f'    [BLOCKER] Significant drift detected - requires recalibration')

gate4_pass = psi < psi_threshold
print(f'\nGate 4 Status: {"[PASS]" if gate4_pass else "[BLOCKER]"}' )

## Gate 5: Diagnostics Validation

In [ ]:
print('\n' + '='*70)
print('GATE 5: DIAGNOSTICS VALIDATION')
print('='*70)

diagnostics = {
    'Data completeness': 100.0,
    'Score validity': 100.0,
    'Distribution stability': 98.5,
    'Anomaly detection': 0,
}

print(f'\nDiagnostics Results:')
all_pass = True
for check, value in diagnostics.items():
    if isinstance(value, float):
        status = '[OK]' if value >= 95.0 else '[ALERT]'
        if value < 95.0:
            all_pass = False
        print(f'  {check:<30}: {value:.1f}% {status}')
    else:
        status = '[OK]' if value == 0 else '[ALERT]'
        if value != 0:
            all_pass = False
        print(f'  {check:<30}: {value} {status}')

gate5_pass = all_pass
print(f'\nGate 5 Status: {"[PASS]" if gate5_pass else "[ALERT]"}' )

## Final Deployment Decision

In [ ]:
print('\n' + '='*70)
print('DEPLOYMENT GATES SUMMARY')
print('='*70)

gates = {
    'Gate 1: Input Validation': gate1_pass,
    'Gate 2: Output Validation': gate2_pass,
    'Gate 3: Distribution Shift': gate3_pass,
    'Gate 4: PSI Detection': gate4_pass,
    'Gate 5: Diagnostics': gate5_pass,
}

print(f'\nGate Results:')
passed_count = 0
for gate, result in gates.items():
    status = '[PASS]' if result else '[FAIL/ALERT]'
    print(f'  {gate:<35} {status}')
    if result:
        passed_count += 1

all_pass = passed_count == len(gates)

print(f'\n' + '='*70)
if all_pass:
    print(f'DEPLOYMENT APPROVED - All gates passed')
    exit_code = 0
else:
    print(f'DEPLOYMENT BLOCKED - {len(gates) - passed_count} gate(s) failed')
    exit_code = 3
print(f'Exit Code: {exit_code}')
print(f'='*70)